# Food Long-Tail OOD Challenge — Vanilla ResNet Baseline

Plainest possible baseline:
1. ResNet-50 ImageNet pretrained, fine-tune on the full training set (no long-tail tricks).
2. Predict the argmax over 101 known classes for every test image.
3. Write `submission.csv`.

Expected behavior: the model can never predict the `unknown` class (101), so the ~25% OOD test samples will all be wrong — capping the achievable Top-1 accuracy at roughly 75%. This baseline establishes the floor; better submissions add long-tail and OOD handling on top.

## 1. Setup

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# 数据根目录与提交输出路径
DATA_ROOT = Path('/Users/hcwu/code/kaggle/food_lt_ood_challenge')
OUT_SUB = DATA_ROOT / 'submission.csv'

# 已知类数：模型只学 0..100，OOD（101）由后处理判定
NUM_KNOWN = 101
IMG_SIZE = 224          # ResNet 默认输入；图像本身是 256，会被 resize 到 224
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 5              # baseline 跑通即可，不追求收敛
LR = 1e-3
SEED = 42

# 自动选设备：优先 CUDA → Apple MPS → CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

torch.manual_seed(SEED); np.random.seed(SEED)

## 2. Dataset

In [ ]:
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])


class FoodDataset(Dataset):
    """从 csv (image_id[, label]) + 图像目录加载样本。
    has_label=False 用于测试集，__getitem__ 第二项返回 image_id 以便回填。"""

    def __init__(self, df, img_dir, transform, has_label):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')
        img = self.transform(img)
        if self.has_label:
            return img, int(row['label'])
        return img, row['image_id']


train_df = pd.read_csv(DATA_ROOT / 'train.csv')
# 借用 test.csv 拿到全部 test image_id（顺序就是提交顺序）
test_df = pd.read_csv(DATA_ROOT / 'test.csv')
print(f'train: {len(train_df)}, test: {len(test_df)}')

tr_ds = FoodDataset(train_df, DATA_ROOT / 'train_images', train_tf, has_label=True)
test_ds = FoodDataset(test_df, DATA_ROOT / 'test_images', eval_tf, has_label=False)

# pin_memory 仅在 CUDA 下能加速，MPS/CPU 无效
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
# 推理时 batch 翻倍，无需反向传播显存压力小
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

## 3. Model

In [ ]:
# ResNet-50 + ImageNet V2 权重（精度比 V1 高约 1%）
# 替换 fc 层，输出维度改为 101（已知类数）
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_KNOWN)
model = model.to(DEVICE)
print(f'ResNet-50, output dim {NUM_KNOWN}')

## 4. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, correct, loss_sum = 0, 0, 0.0
    bar = tqdm(tr_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False)
    for x, y in bar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
        bar.set_postfix(loss=loss_sum/total, acc=correct/total)
    scheduler.step()
    print(f'Epoch {epoch}/{EPOCHS} | loss {loss_sum/total:.3f} | train acc {correct/total:.3f} | {time.time()-t0:.1f}s')

## 5. Predict on test set

In [ ]:
# 朴素 OOD 检测：MSP (Maximum Softmax Probability)
# 若 softmax 最大概率低于阈值，判为 unknown (101)


model.eval()
all_ids, all_preds = [], []
with torch.no_grad():
    for x, ids in tqdm(test_loader, desc='predict'):
        x = x.to(DEVICE, non_blocking=True)
        probs = model(x).softmax(dim=1)
        msp, pred = probs.max(dim=1)         # msp: 最大类概率, pred: argmax
        # 低置信样本改判为 OOD
        pred = torch.where(msp < MSP_THRESHOLD,
                           torch.full_like(pred, OOD_LABEL),
                           pred)
        all_preds.extend(pred.cpu().numpy().tolist())
        all_ids.extend(ids)
print(f'Predicted {len(all_ids)} samples')

## 6. Submission

In [ ]:
sub = pd.DataFrame({'image_id': all_ids, 'label': all_preds})
# DataLoader 是 shuffle=False，但 num_workers>0 时返回顺序不保证完全一致
# 这里按 sample_submission.csv 的 image_id 顺序重排，确保提交对齐
order = pd.read_csv(DATA_ROOT / 'test.csv')['image_id'].tolist()
sub = sub.set_index('image_id').loc[order].reset_index()
sub.to_csv(OUT_SUB, index=False)
print(f'Wrote {OUT_SUB} ({len(sub)} rows)')
sub.head()